In [1]:
# Install DeepMReye
!pip install deepmreye

In [2]:
import os
import nibabel as nib
import numpy as np
import glob

from collections import defaultdict
from deepmreye import preprocess, train, analyse
from deepmreye.util import model_opts, data_generator
from deepmreye.preprocess import get_masks, run_participant, save_data, normalize_img
from deepmreye.architecture import create_standard_model
from deepmreye.analyse import visualise_predictions_slider
from sklearn.model_selection import train_test_split
from nibabel import load
from funcs import visualise_predictions_slider, get_test_subject_data

2025-11-03 00:06:25.997183: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-11-03 00:06:26.046460: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-11-03 00:06:26.046493: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-11-03 00:06:26.048335: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-11-03 00:06:26.055817: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-11-03 00:06:26.058193: I tensorflow/core/platform/cpu_feature_guard.cc:1

In [3]:
# Define paths
base_path = "/data2/2104/derivatives/deepmreye/"
calibration_npz = os.path.join(base_path, "calibration_npz")
movie_npz = os.path.join(base_path, "movie_npz")
model_weights = os.path.join(base_path, "model_weights")

In [15]:
opts = model_opts.get_opts()
model, model_inference = create_standard_model(input_shape=(47, 29, 18, 1), opts=opts)
model.load_weights(model_weights + '/dataset5_free_viewing.h5')
#model.load_weights(model_weights + '/datasets_1to5.h5')
model_inference.set_weights(model.get_weights())

calib_files = sorted(glob.glob(os.path.join(calibration_npz, "*.npz")))
movie_files = sorted(glob.glob(os.path.join(movie_npz, "*.npz")))

g_train = data_generator.data_generator(
    calib_files, 
    opts['batch_size'], 
    training=True, 
    mixed_batches=True,
    withinsubject_split=[0.0, 0.8]  # Train on first 80%
)
        
g_val = data_generator.data_generator(
    calib_files, 
    opts['batch_size'], 
    training=False, 
    mixed_batches=True,
    withinsubject_split=[0.8, 1.0]  # Validate on last 20%
)

gens_finetune = (
    g_train,               
    g_val,           
    [], [], [], [],      
    calib_files,    
    calib_files,         
)

# Define options for model training
opts = model_opts.get_opts()
opts['epochs'] = 1
opts['steps_per_epoch'] = 1000
opts['validation_steps'] = 200

# Train model
model, model_inference = train.train_model(
    dataset=f"refine", generators=gens_finetune, opts=opts,
    models=(model, model_inference), clear_graph=False, save=False, verbose=0
)

1000/1000 [==============================] - 2312s 2s/step - loss: 2.8192 - val_loss: 1.7064 - lr: 2.0000e-05


In [16]:
def _load_all_X_unlabeled(npz_path):
    with np.load(npz_path, mmap_mode="r") as z:
        data_keys = sorted(
            (k for k in z.files if k.startswith("data_")),
            key=lambda k: int(k.split("_")[1])
        )
        X = [z[k] for k in data_keys]
    return np.asarray(X)[..., np.newaxis] 


def evaluate_unlabeled(model_inference, movie_files, batch_size=16, verbose=1, save_dir=None):
    evaluation, scores = {}, {}
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)

    for idx, path in enumerate(movie_files):
        X = _load_all_X_unlabeled(path)
        pred_y, euc_pred = model_inference.predict(X, batch_size=batch_size, verbose=max(0, verbose-1))
        real_y = np.full_like(pred_y, np.nan)  # keep same structure as labeled
        evaluation[path] = {"real_y": real_y, "pred_y": pred_y, "euc_pred": euc_pred}
        scores[path] = None  # no metrics without labels

        if save_dir:
            out = os.path.join(save_dir, os.path.basename(path).replace(".npz", "_pred.npz"))
            np.savez_compressed(out, pred_y=pred_y, euc_pred=euc_pred)

        if verbose:
            print(f"{idx+1}/{len(movie_files)} predicted: {path}  -> {pred_y.shape[0]} timesteps")

    return evaluation, scores


evaluation, scores = evaluate_unlabeled(model_inference, movie_files, batch_size=16, verbose=1)

1/116 predicted: /data2/2104/derivatives/deepmreye/movie_npz/sub-21040005_ses-3_run-1.npz  -> 1634 timesteps
2/116 predicted: /data2/2104/derivatives/deepmreye/movie_npz/sub-21040005_ses-3_run-2.npz  -> 1685 timesteps
3/116 predicted: /data2/2104/derivatives/deepmreye/movie_npz/sub-21040005_ses-3_run-3.npz  -> 1592 timesteps
4/116 predicted: /data2/2104/derivatives/deepmreye/movie_npz/sub-21040005_ses-4_run-1.npz  -> 1687 timesteps
5/116 predicted: /data2/2104/derivatives/deepmreye/movie_npz/sub-21040005_ses-4_run-2.npz  -> 1637 timesteps
6/116 predicted: /data2/2104/derivatives/deepmreye/movie_npz/sub-21040005_ses-4_run-3.npz  -> 1498 timesteps
7/116 predicted: /data2/2104/derivatives/deepmreye/movie_npz/sub-21040010_ses-3_run-1.npz  -> 1525 timesteps
8/116 predicted: /data2/2104/derivatives/deepmreye/movie_npz/sub-21040010_ses-3_run-2.npz  -> 1525 timesteps
9/116 predicted: /data2/2104/derivatives/deepmreye/movie_npz/sub-21040010_ses-3_run-3.npz  -> 1656 timesteps
10/116 predicted: /

In [ ]:
fig = visualise_predictions_slider(evaluation, {},
                                   color="rgb(0, 150, 175)", bg_color="rgb(255,255,255)", ylim=[-11, 11])
fig.show()

In [ ]:
# # Detailed array contents - NO TRUNCATION
# txt_path = os.path.join(base_path, "combined_eval_detailed.txt")

# with open(txt_path, 'w') as f:
#     f.write("DETAILED COMBINED_EVAL ARRAY CONTENTS:\n")
#     f.write("=" * 50 + "\n\n")
    
#     for run_id, data in combined_eval.items():
#         f.write(f"RUN: {run_id}\n")
#         f.write(f"pred_y shape: {data['pred_y'].shape}\n")
#         f.write(f"euc_pred shape: {data['euc_pred'].shape}\n")
#         f.write(f"real_y shape: {data['real_y'].shape}\n")
#         f.write("-" * 40 + "\n")
        
#         f.write("PREDICTIONS (pred_y) - ALL VALUES:\n")
#         # Reshape 3D array to 2D for saving
#         flattened_pred = data['pred_y'].reshape(-1, data['pred_y'].shape[-1])
#         np.savetxt(f, flattened_pred, fmt='%.6f')
#         f.write("\n")
        
#         f.write("EUCLIDEAN DISTANCES (euc_pred):\n")
#         np.savetxt(f, data['euc_pred'], fmt='%.6f')
#         f.write("\n")
        
#         f.write("GROUND TRUTH (real_y) - ALL VALUES:\n")
#         # Reshape 3D array to 2D for saving
#         flattened_real = data['real_y'].reshape(-1, data['real_y'].shape[-1])
#         np.savetxt(f, flattened_real, fmt='%.6f')
#         f.write("\n" + "=" * 50 + "\n\n")

# print(f"Detailed array contents saved to: {txt_path}")

In [17]:
# Save median predictions per TR (no real_y)
txt_path = os.path.join(base_path, "evals/eval_median_5_v2_sub-TR.txt")

with open(txt_path, 'w') as f:
    for run_id, data in evaluation.items():
        # Calculate median across samples for each TR
        median_pred = np.median(data['pred_y'], axis=1)  # Shape: (1634, 2)
        
        f.write(f"RUN: {run_id}\n")
        f.write("TR\tX\t\tY\n")
        for tr_idx, (x, y) in enumerate(median_pred):
            f.write(f"{tr_idx+1}\t{x:.6f}\t{y:.6f}\n")
        f.write("\n\n")

print(f"Median predictions per TR saved to: {txt_path}")

Median predictions per TR saved to: /data2/2104/derivatives/deepmreye/evals/eval_median_5_v2_sub-TR.txt
